# Módulo 13: Historia, contexto y definiciones — Guía del ejercicio

En este ejercicio vas a aplicar el proceso KDD (descubrimiento de conocimiento en bases de datos) a un caso de negocio. Primero lo razonarás por escrito: los pasos del proceso, las aplicaciones de la minería de datos y sus retos. Después recorrerás un mini-proceso KDD en Python sobre tickets de compra, desde la selección de los datos hasta una recomendación basada en reglas de asociación.

- Si necesitas repasar los conceptos, consulta la lección [m13-historia-contexto-y-definiciones-leccion.md](m13-historia-contexto-y-definiciones-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m13-historia-contexto-y-definiciones-solucion.ipynb](m13-historia-contexto-y-definiciones-solucion.ipynb).

## Contexto

**Supermercados La Milpa** es una cadena regional con tres sucursales: Centro, Lomas y Valle. Sus cajas registran cada venta desde hace años, pero los tickets solo se usan para cuadrar la caja al final del día. La nueva gerente de mercadotecnia quiere aprovecharlos para decidir **qué productos promocionar juntos y cómo acomodar los anaqueles**.

El área de sistemas te entrega una extracción de los tickets de julio a septiembre de 2025. Te advierten que los datos "no están muy limpios": cada sucursal captura los nombres de producto a su manera, algunas líneas se registraron dos veces por fallas del sistema de cobro y la extracción incluye devoluciones.

## Datos

La celda inicial genera el DataFrame `tickets`, con una fila por línea de ticket:

| Columna | Descripción |
| --- | --- |
| `ticket_id` | Identificador del ticket (una compra) |
| `sucursal` | Sucursal donde se hizo la compra: `Centro`, `Lomas` o `Valle` |
| `fecha` | Fecha de la compra |
| `producto` | Nombre del producto tal como lo capturó la caja (con variantes de mayúsculas, acentos y espacios) |
| `cantidad` | Unidades vendidas; un valor negativo indica una devolución |

## Instrucciones

- Las tareas 1 a 3 son preguntas escritas sobre el caso. Respóndelas en la celda de texto que sigue a cada una, con tus palabras y apoyándote en la lección.
- Las tareas 4 a 7 son la práctica en Python. Escribe tu código en las celdas vacías y añade más celdas si lo necesitas.
- Acompaña cada resultado numérico con una frase breve que lo interprete.
- La celda siguiente importa las librerías y simula la extracción de tickets con una semilla fija. Trátala como la base de datos de la empresa: no la modifiques y descubre los patrones con el análisis, no leyendo el código que los genera.

In [ ]:
import re
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

variantes = [
    ["Leche Entera", "LECHE ENTERA", "leche entera ", " Leche  entera"],
    ["Pan de Caja", "PAN DE CAJA", "pan de caja", "Pan  de caja "],
    ["Huevo", "HUEVO", " huevo"],
    ["Café Soluble", "CAFE SOLUBLE", "cafe soluble", "Café  soluble "],
    ["Azúcar", "AZUCAR", "azucar ", "Azucar"],
    ["Cerveza", "CERVEZA", "cerveza "],
    ["Papas Fritas", "PAPAS FRITAS", "papas  fritas", "PAPAS FRIT."],
    ["Refresco", "REFRESCO", "refresco "],
    ["Tortillas", "TORTILLAS", " tortillas"],
    ["Detergente", "DETERGENTE", "detergente "],
    ["Suavizante", "SUAVIZANTE", "Suavizante "],
    ["Frijol", "FRIJOL", "frijol "],
]
prob = np.array([0.45, 0.30, 0.28, 0.12, 0.08, 0.15, 0.09, 0.25, 0.30, 0.08, 0.03, 0.15])
n_tickets = 3000

compra = rng.random((n_tickets, len(prob))) < prob
for a, b, p in [(3, 4, 0.55), (5, 6, 0.6), (9, 10, 0.45)]:
    compra[:, b] |= compra[:, a] & (rng.random(n_tickets) < p)
vacios = ~compra.any(axis=1)
compra[vacios, rng.integers(0, len(prob), vacios.sum())] = True

t_idx, p_idx = np.nonzero(compra)
tickets = pd.DataFrame({
    "ticket_id": 100000 + t_idx,
    "sucursal": np.array(["Centro", "Lomas", "Valle"])[rng.integers(0, 3, n_tickets)][t_idx],
    "fecha": (pd.Timestamp("2025-07-01") + pd.to_timedelta(rng.integers(0, 92, n_tickets), unit="D"))[t_idx],
    "producto": [variantes[j][rng.integers(len(variantes[j]))] for j in p_idx],
    "cantidad": rng.integers(1, 4, len(t_idx)),
})
dobles = tickets.sample(frac=0.05, random_state=42)
dobles["producto"] = [variantes[j][rng.integers(len(variantes[j]))] for j in p_idx[dobles.index]]
devoluciones = tickets.sample(n=60, random_state=7).assign(cantidad=lambda d: -d["cantidad"])
devoluciones["ticket_id"] = 200000 + np.arange(60)
tickets = pd.concat([tickets, dobles, devoluciones], ignore_index=True)
tickets.loc[rng.choice(len(tickets), 15, replace=False), "producto"] = np.nan
tickets = (tickets.sample(frac=1, random_state=42)
           .sort_values("ticket_id", kind="stable")
           .reset_index(drop=True))

print(tickets.shape)
tickets.head(10)

## Tarea 1. El proceso KDD en La Milpa

La lección describe el KDD en seis pasos: comprender el dominio, seleccionar los datos, prepararlos, aplicar minería de datos, interpretar y evaluar, y usar el conocimiento descubierto. En la práctica de este ejercicio, la **transformación** de los datos (que la lección incluye dentro de la preparación) aparece como un paso aparte, porque cambia la forma de la tabla.

Para cada uno de los seis pasos, explica en una o dos frases qué significa **en el caso de La Milpa**: qué pregunta se responde, qué datos o decisiones intervienen y qué se obtiene al terminar el paso. Evita definiciones genéricas: cada respuesta debe mencionar algo concreto del caso (los tickets, las sucursales, las devoluciones, la gerente de mercadotecnia…).

*Escribe aquí tu respuesta.*

## Tarea 2. Aplicaciones pertinentes

La lección menciona aplicaciones de la minería de datos en varias industrias, entre ellas el comercio: segmentación de clientes, previsión de ventas y análisis de riesgo.

1. Propón **tres aplicaciones** de la minería de datos que La Milpa podría desarrollar con sus tickets (o con datos que podría empezar a reunir).
2. Para cada una indica: el problema de negocio que resuelve, los datos que necesita y la técnica de minería que usarías (asociación, segmentación o *clustering*, regresión, clasificación…).
3. Indica cuál de las tres harías primero y por qué.

*Escribe aquí tu respuesta.*

## Tarea 3. Retos concretos

A partir de los retos que describe la lección (escalabilidad, calidad e integración de los datos, comprensibilidad de los modelos, privacidad y ética, formación), identifica **al menos cuatro retos concretos** que La Milpa enfrentará en este proyecto. Incluye como mínimo uno de cada uno de estos temas: **calidad de los datos**, **privacidad** y **volumen**. Para cada reto, describe cómo se manifiesta en el caso y una medida para mitigarlo.

*Escribe aquí tu respuesta.*

## Tarea 4. Selección y preparación de los datos

1. **Selección.** Crea un DataFrame `ventas` que conserve solo las líneas de venta (`cantidad > 0`) y las columnas `ticket_id` y `producto`. Indica cuántas líneas descartaste y por qué no deben entrar en un análisis de canastas.
2. **Limpieza.** Elimina las filas sin producto y cuenta cuántas eran.
3. **Normalización.** Antes de normalizar, cuenta cuántos nombres de producto distintos hay y cuántas filas están duplicadas (mismo `ticket_id` y mismo `producto`). Después escribe una función `normalizar(nombre)` que pase el texto a minúsculas, quite los acentos (con `unicodedata.normalize("NFKD", ...)`), elimine los espacios al inicio y al final y reduzca los espacios repetidos a uno (con `re.sub`). Aplícala a `producto` y compara el número de nombres distintos antes y después.
4. Revisa la lista de nombres resultante. Si algún producto todavía aparece con dos nombres, unifícalos con un diccionario y `replace`. Al final deben quedar **12 productos distintos**.
5. **Duplicados.** Cuenta de nuevo las filas duplicadas, compara con el conteo anterior a la normalización y elimínalas. Explica por qué la normalización revela duplicados que antes no se veían.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 5. Transformación: la canasta en formato *one-hot*

1. Convierte `ventas` en una matriz de canastas `canasta`: una fila por ticket, una columna por producto y un valor `True` si el ticket contiene el producto (puedes usar `pd.crosstab` y comparar con `> 0`).
2. Verifica las dimensiones (número de tickets × 12) y que ningún ticket quedó vacío.
3. Calcula el **soporte** de cada producto, es decir, la proporción de tickets que lo contienen, y represéntalo en un gráfico de barras ordenado de mayor a menor.

$$\text{soporte}(A) = \frac{\text{tickets que contienen } A}{\text{total de tickets}}$$

4. Comenta qué productos son los más y los menos frecuentes.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 6. Minería: soporte, confianza y *lift* de pares de productos

Para cada par **ordenado** de productos distintos $(A, B)$, la regla $A \rightarrow B$ ("quien compra $A$ también compra $B$") se mide con:

$$\text{soporte}(A, B) = \frac{\text{tickets con } A \text{ y } B}{\text{total de tickets}} \qquad \text{confianza}(A \rightarrow B) = \frac{\text{soporte}(A, B)}{\text{soporte}(A)} \qquad \text{lift}(A \rightarrow B) = \frac{\text{confianza}(A \rightarrow B)}{\text{soporte}(B)}$$

Un *lift* mayor que 1 indica que $A$ y $B$ aparecen juntos más de lo que se esperaría si se compraran de forma independiente.

1. Calcula el número de tickets que contienen cada par de productos (pista: con una matriz de 0 y 1, el producto `canasta.T @ canasta` da todos los conteos a la vez).
2. Construye una tabla `reglas` con las columnas `antecedente`, `consecuente`, `soporte`, `confianza` y `lift`, con una fila por par ordenado (12 × 11 = 132 reglas).
3. Conserva las reglas con soporte de al menos 0.02 y muestra las 6 de mayor *lift*.
4. Muestra también las 6 reglas de mayor **confianza**. ¿Hay reglas con confianza alta pero *lift* cercano a 1? Explica por qué la confianza sola puede engañar.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Tarea 7. Interpretación: de las reglas a una recomendación

1. Haz un gráfico de barras horizontales con el *lift* de las reglas que conservaste en la Tarea 6 cuyo *lift* sea mayor que 1.5, con una línea vertical de referencia en *lift* = 1.
2. Redacta una **recomendación de negocio** para la gerente de mercadotecnia, de tres a cinco líneas: qué pares de productos aprovechar y con qué acción concreta (promoción conjunta, acomodo en anaquel, cupón…).
3. Menciona una **limitación** de este análisis que la gerente debería conocer antes de actuar.

In [ ]:
# Tu código aquí

*Escribe aquí tu respuesta.*

## Antes de terminar

- [ ] Relacionaste cada uno de los seis pasos del KDD con algo concreto del caso de La Milpa.
- [ ] Propusiste tres aplicaciones con su problema, sus datos y su técnica, y elegiste una para empezar.
- [ ] Identificaste retos de calidad de los datos, privacidad y volumen, cada uno con una medida de mitigación.
- [ ] Tu tabla `ventas` limpia tiene exactamente 12 productos distintos y ningún duplicado.
- [ ] La matriz `canasta` tiene una fila por ticket y una columna por producto.
- [ ] La tabla `reglas` tiene 132 filas y las columnas `antecedente`, `consecuente`, `soporte`, `confianza` y `lift`.
- [ ] Tu recomendación se apoya en las reglas con mayor *lift* e incluye una limitación.

Cuando termines, compara tu trabajo con la solución: [m13-historia-contexto-y-definiciones-solucion.ipynb](m13-historia-contexto-y-definiciones-solucion.ipynb).